# Target EDA: Product Acquisitions

This notebook creates disk-backed acquisition labels, then analyses new-product purchases.

## Bootstrap

In [ ]:
from __future__ import annotations

import base64
import binascii
import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

REPO_OWNER = 'lbngyn'
REPO_NAME = 'Santander-Product-Recommendation'
REPO_BRANCH = 'feat/colab-local-run'
REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
RUNTIME_CONFIG_KEYS = ('GITHUB_TOKEN', 'GCP_SERVICE_ACCOUNT_JSON', 'GOOGLE_CLOUD_PROJECT', 'GCS_BUCKET', 'GCS_RAW_PREFIX', 'GCS_CHECKPOINT_PREFIX')

def is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise FileNotFoundError('Cannot find project root containing src/ and requirements.txt.')

def run_git(arguments: list[str], token: str) -> None:
    credentials = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    subprocess.run(['git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {credentials}', *arguments], check=True)

def load_colab_runtime_config() -> dict[str, str]:
    try:
        from google.colab import userdata
        values = {key: userdata.get(key) for key in RUNTIME_CONFIG_KEYS}
        if all(values.values()):
            return {key: str(value) for key, value in values.items()}
    except Exception:
        pass
    encoded = os.getenv('COLAB_RUNTIME_CONFIG_B64') or getpass('Paste COLAB_RUNTIME_CONFIG_B64 once (runtime-only): ')
    try:
        values = json.loads(base64.b64decode(encoded, validate=True).decode('utf-8'))
    except (binascii.Error, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError('COLAB_RUNTIME_CONFIG_B64 must be a Base64-encoded JSON object.') from error
    missing = [key for key in RUNTIME_CONFIG_KEYS if not values.get(key)]
    if missing:
        raise ValueError(f'Runtime config bundle is missing: {missing}')
    if isinstance(values['GCP_SERVICE_ACCOUNT_JSON'], dict):
        values['GCP_SERVICE_ACCOUNT_JSON'] = json.dumps(values['GCP_SERVICE_ACCOUNT_JSON'])
    return {key: str(values[key]) for key in RUNTIME_CONFIG_KEYS}

IS_COLAB = is_colab_runtime()
ENV = 'colab' if IS_COLAB else 'local'
os.environ['SANTANDER_RUNTIME'] = ENV

if IS_COLAB:
    from google.colab import drive
    runtime_config = load_colab_runtime_config()
    os.environ.update(runtime_config)
    PROJECT_ROOT = Path('/content') / REPO_NAME
    if not PROJECT_ROOT.exists():
        run_git(['clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], runtime_config['GITHUB_TOKEN'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)
    drive_mount = Path('/content/drive')
    if not (drive_mount / 'MyDrive').exists():
        drive.mount(str(drive_mount))
    DATA_ROOT = drive_mount / 'MyDrive/projects/santander/data'
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    load_dotenv(PROJECT_ROOT / '.env')
    DATA_ROOT = Path(os.getenv('SANTANDER_DATA_ROOT', PROJECT_ROOT / 'data'))

os.environ['SANTANDER_DATA_ROOT'] = str(DATA_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Notebook ready | environment={ENV} | source={PROJECT_ROOT}')


## Create acquisition checkpoint

In [ ]:
from src.data.ingest import config_from_environment, run_default_ingests
from src.features.acquisition import build_acquisition_features

INGEST_CONFIG = config_from_environment()
run_default_ingests(config=INGEST_CONFIG)

feature_result = build_acquisition_features(
    train_path=INGEST_CONFIG.checkpoint_dir / 'train.parquet',
    test_path=INGEST_CONFIG.checkpoint_dir / 'test.parquet',
    processed_dir=INGEST_CONFIG.checkpoint_dir.parent / 'processed',
    memory_limit='2GB' if ENV == 'local' else '8GB',
    temp_directory=DATA_ROOT / 'duckdb_temp',
)
feature_result


## Setup

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

TRAIN_PARQUET = Path(feature_result['train'])
TEST_PARQUET = Path(feature_result['test']) if feature_result['test'] else None
DUCKDB_TEMP_DIR = DATA_ROOT / 'duckdb_temp'
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(database=':memory:')
con.execute(f"SET temp_directory = '{str(DUCKDB_TEMP_DIR).replace(chr(39), chr(39) * 2)}'")
con.execute("SET memory_limit = '2GB'" if ENV == 'local' else "SET memory_limit = '8GB'")

def q(sql: str, *params: object) -> pd.DataFrame:
    return con.execute(sql, list(params)).df()

def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

ALL_COLUMNS = q('DESCRIBE SELECT * FROM read_parquet(?)', str(TRAIN_PARQUET))['column_name'].tolist()
PRODUCT_COLUMNS = [column for column in ALL_COLUMNS if column.endswith('_ult1')]
ACQUISITION_COLUMNS = [f'acq_{column}' for column in PRODUCT_COLUMNS]
assert set(ACQUISITION_COLUMNS).issubset(ALL_COLUMNS), 'Acquisition checkpoint is incomplete.'


In [ ]:
processed_shape = q('SELECT COUNT(*) AS rows FROM read_parquet(?)', str(TRAIN_PARQUET))
processed_shape['columns'] = len(ALL_COLUMNS)
display(processed_shape)
display(q('DESCRIBE SELECT * FROM read_parquet(?)', str(TRAIN_PARQUET)))


## Acquisition EDA

In [ ]:
# New-purchase distribution by product type.
product_acquisition_sql = ' UNION ALL '.join(
    f"SELECT '{product}' AS product, SUM({quote_identifier(acquisition)})::BIGINT AS new_purchase_count FROM read_parquet(?)"
    for product, acquisition in zip(PRODUCT_COLUMNS, ACQUISITION_COLUMNS)
)
product_acquisition = q(product_acquisition_sql, *([str(TRAIN_PARQUET)] * len(PRODUCT_COLUMNS))).sort_values('new_purchase_count', ascending=False)
display(product_acquisition)
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(product_acquisition['product'], product_acquisition['new_purchase_count'], color='#1f77b4')
ax.set(title='Phân phối mua mới của từng loại sản phẩm', xlabel='Loại sản phẩm', ylabel='Số lượng sản phẩm mua mới')
ax.tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of total new purchases per customer across the full history.
total_acquisition_expression = ' + '.join(quote_identifier(column) for column in ACQUISITION_COLUMNS)
summed_acquisition_expression = ' + '.join(f'SUM({quote_identifier(column)})' for column in ACQUISITION_COLUMNS)
total_new_purchases = q(f'''
    WITH customer_totals AS (
        SELECT ncodpers, ({summed_acquisition_expression}) AS total_new_purchases
        FROM read_parquet(?)
        GROUP BY ncodpers
    )
    SELECT total_new_purchases, COUNT(*) AS customer_count
    FROM customer_totals
    GROUP BY total_new_purchases
    ORDER BY total_new_purchases
''', str(TRAIN_PARQUET))
display(total_new_purchases)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(total_new_purchases['total_new_purchases'], total_new_purchases['customer_count'], color='#ff7f0e')
ax.set(title='Phân bố tổng số mua mới trên mỗi khách hàng', xlabel='Tổng số sản phẩm mua mới', ylabel='Số khách hàng')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of new products purchased by a customer in one month.
monthly_new_purchases = q(f'''
    WITH customer_month_totals AS (
        SELECT ncodpers, fecha_dato, ({summed_acquisition_expression}) AS monthly_new_purchases
        FROM read_parquet(?)
        GROUP BY ncodpers, fecha_dato
    )
    SELECT monthly_new_purchases, COUNT(*) AS customer_month_count
    FROM customer_month_totals
    GROUP BY monthly_new_purchases
    ORDER BY monthly_new_purchases
''', str(TRAIN_PARQUET))
display(monthly_new_purchases)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(monthly_new_purchases['monthly_new_purchases'], monthly_new_purchases['customer_month_count'], color='#2ca02c')
ax.set(title='Phân bố số mua mới trong một tháng trên mỗi khách hàng', xlabel='Số sản phẩm mua mới / khách hàng-tháng', ylabel='Số khách hàng-tháng')
plt.tight_layout()
plt.show()


In [ ]:
# One time-series chart per product shows whether the sales peak is shared or product-specific.
monthly_product_acquisition_sql = ' UNION ALL '.join(
    f"SELECT fecha_dato, '{product}' AS product, SUM({quote_identifier(acquisition)})::BIGINT AS new_purchase_count FROM read_parquet(?) GROUP BY fecha_dato"
    for product, acquisition in zip(PRODUCT_COLUMNS, ACQUISITION_COLUMNS)
)
monthly_product_acquisition = q(
    monthly_product_acquisition_sql,
    *([str(TRAIN_PARQUET)] * len(PRODUCT_COLUMNS)),
).sort_values(['product', 'fecha_dato'])

fig, axes = plt.subplots(6, 4, figsize=(18, 18), sharex=True)
for ax, product in zip(axes.flat, PRODUCT_COLUMNS):
    data = monthly_product_acquisition[monthly_product_acquisition['product'] == product]
    ax.plot(data['fecha_dato'], data['new_purchase_count'], linewidth=1.5)
    ax.set_title(product, fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.set_ylabel('Số mua mới', fontsize=8)
for ax in axes.flat[len(PRODUCT_COLUMNS):]:
    ax.set_visible(False)
fig.suptitle('Phân phối mua mới theo thời gian — từng loại sản phẩm', y=1.01)
plt.tight_layout()
plt.show()
